In [ ]:
import dxpy, dxdata
import subprocess
import os
import subprocess
from datetime import datetime
import gzip, shutil
import pandas as pd
import numpy as np
import re
import ast

In [ ]:
# --- Initialise database ---
def init_db():
    dispensed_database = dxpy.find_one_data_object(
        classname='database', 
        name='app*', 
        folder='/', 
        name_mode='glob', 
        describe=True)
    dispensed_database_name = dispensed_database['describe']['name']
    dispensed_dataset = dxpy.find_one_data_object(
        typename='Dataset', 
        name='app*.dataset', 
        folder='/', 
        name_mode='glob')
    dispensed_dataset_id = dispensed_dataset['id']
    dataset = dxdata.load_dataset(id=dispensed_dataset_id)
    participant = dataset['participant']
    return(participant)

# --- Resolver (yours) ---
def real_field_names_for_showcase_id(field_id: int):
    field_id = str(field_id)
    fields = participant.find_fields(name_regex=rf"^p{field_id}(_i\d+)?(_a\d+)?$")
    return sorted([f.name for f in fields])

# --- dx helpers ---
def run_dx(cmd):
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr.strip() or "dx failed with no stderr")
    return p

def write_fields_file(path, entity, cols):
    with open(path, "w") as f:
        for c in cols:
            f.write(f"{entity}.{c}\n")

def gzip_inplace(path):
    gz = path + ".gz"
    with open(path, "rb") as f_in, gzip.open(gz, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(path)
    return gz

def chunk_list(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

# --- Main: resolve -> column chunks -> extract ---
def extract_by_resolved_columns(
    participant,
    dataset_fqid: str,
    showcase_ids,
    entity: str = "participant",
    cols_per_chunk: int = 25,      # tune this down if you still get DataTooLarge
    delimiter: str = "\t",
    out_stem: str = "ukb_extract",
):
    # 1) Resolve to dataset columns
    resolved = {fid: real_field_names_for_showcase_id(fid) for fid in showcase_ids}

    missing = [fid for fid, cols in resolved.items() if not cols]
    if missing:
        print("Missing/unavailable showcase IDs (skipped):", missing)

    # Flatten, de-duplicate, keep order; always include eid
    cols = ["eid"]
    for fid in showcase_ids:
        cols.extend(resolved.get(fid, []))

    seen = set()
    cols = [c for c in cols if not (c in seen or seen.add(c))]

    print(f"Resolved {len(cols)} dataset columns (including eid).")

    # 2) Chunk by columns (excluding eid), include eid in every extract
    non_eid = [c for c in cols if c != "eid"]
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{out_stem}.{ts}"
    outputs = []

    for part_idx, col_chunk in enumerate(chunk_list(non_eid, cols_per_chunk)):
        part_cols = ["eid"] + col_chunk

        fields_file = f"{prefix}.part{part_idx:04d}.fields.txt"
        out_tsv = f"{prefix}.part{part_idx:04d}.tsv"

        write_fields_file(fields_file, entity, part_cols)

        try:
            run_dx([
                "dx", "extract_dataset", dataset_fqid,
                "--fields-file", fields_file,
                "--delimiter", delimiter,
                "-o", out_tsv
            ])
        except RuntimeError as e:
            # If this still happens, reduce cols_per_chunk or subset rows via a Cohort
            raise RuntimeError(f"Part {part_idx} failed (try smaller cols_per_chunk):\n{e}")

        outputs.append(gzip_inplace(out_tsv))
        print("Wrote", outputs[-1])

    return resolved, prefix, outputs

In [ ]:
run_dx(["dx", "whoami"])
run_dx(["dx", "pwd"])

dispensed_dataset = dxpy.find_one_data_object(
    typename="Dataset",
    name="app*.dataset",
    folder="/",
    name_mode="glob",
    describe=True
)
dataset_id = dispensed_dataset["id"]

In [ ]:
participant = init_db()
project_id = dxpy.PROJECT_CONTEXT_ID
dataset_fqid = f"{project_id}:{dataset_id}"
showcase_ids = [
    31, 34, 52, 53, 904, 1558, 2335, 2443, 4079, 4080, 6138, 6177, 20116, 21001, 21022, 22006, 22009, 22189, 23099, 26223, 26227, 26250,
    30500, 30510, 30520, 30530, 30620, 30640, 30700, 30710, 30730, 30740, 30750, 30760, 30780, 30790, 30870, 30880
]

In [ ]:
resolved_map, prefix, files = extract_by_resolved_columns(participant, dataset_fqid, showcase_ids, cols_per_chunk=20)
resolved_map, prefix, files

In [ ]:
# Load data from chunks
df_paths = [entry for entry in os.listdir('.') if entry.startswith(prefix) and entry.endswith('.tsv.gz') and os.path.isfile(entry)]

df_list = [pd.read_csv(x, sep='\t', compression='gzip') for x in df_paths]

for df in df_list:
    df.columns = df.columns.str.removeprefix("participant.")
    df.set_index('eid', inplace = True)

data = pd.concat(df_list, axis=1)
data.reset_index(inplace = True)

In [ ]:
data.columns

In [ ]:
# Map UK Biobank phenocodes onto interpretable variable names
pc_map = {f"p22009_a{x}": f"pc{x}" for x in range(1,41)}
cov_map = {
    'eid': 'eid',
    'p31': 'sex', 
    'p34': 'year_of_birth',
    'p52': 'month_of_birth',
    'p53_i0': 'date_assessment',
    'p904_i0': 'physical_activity',
    'p1558_i0': 'alcohol_use',
    'p2335_i0': 'has_chest_pain',
    'p2443_i0': 'has_diabetes',
    'p4079_i0_a1': 'diastolic_bp',
    'p4080_i0_a1': 'systolic_bp',
    'p6138_i0': 'edu_quals',
    'p6177_i0': 'taking_chol_bp_meds',
    'p20116_i0': 'smoking_status',
    'p21001_i0': 'bmi',
    'p21022': 'age_assessment',
    'p22006': 'caucasian',
    'p22189': 'townsend_index',
    'p23099_i0': 'body_fat_percentage',
    'p26223': 'cvd_prs',
    'p26227': 'cad_prs',
    'p26250': 'ldl_prs',
    'p30500_i0': 'urine_microalbumin_level',
    'p30510_i0': 'urine_creatinine_level',
    'p30520_i0': 'urine_potassium_level',
    'p30530_i0': 'urine_sodium_level',
    'p30620_i0': 'alt_level',
    'p30640_i0': 'apob_level',
    'p30700_i0': 'creatinine_level',
    'p30710_i0': 'crp_level',
    'p30730_i0': 'ggt_level',
    'p30740_i0': 'glucose_level',
    'p30750_i0': 'hba1c_level',
    'p30760_i0': 'hdl_level',
    'p30780_i0': 'ldl_level',
    'p30790_i0': 'lpa_level',
    'p30870_i0': 'triglyceride_level',
    'p30880_i0': 'urate_level'
}
mapping = {**cov_map, **pc_map}
data = data.rename(mapping, axis=1)
data = data[mapping.values()]

In [ ]:
# Format the questionnaire answers
questionnaire_cols = ['alcohol_use', 'smoking_status', 'physical_activity', 'has_chest_pain', 'has_diabetes']
data[questionnaire_cols] = data[questionnaire_cols].replace([-1, -3, -7], np.nan)
# data = data[data[questionnaire_cols].notna().all(axis=1)]

In [ ]:
LEVELS = [
    "College or University",
    "A-levels",
    "O-levels",
    "None of the above",
    "Other",
]

# UKB p6138 / qualifications codes
CODE_TO_LEVEL = {
    1: "College or University",
    2: "A-levels",
    3: "O-levels",
    -7: "None of the above",
    # everything else (4,5,6, -3, unknown) -> "Other" or missing depending on choice
}

# precedence when multiple codes are present
PRECEDENCE = {
    "College or University": 0,
    "A-levels": 1,
    "O-levels": 2,
    "None of the above": 3,
    "Other": 4,
}

def parse_qual_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    # already list-like
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        raw = list(x)
    else:
        s = str(x).strip()
        if s.lower() in {"nan", "none", ""}:
            return []
        # try "[2,3,6]" / "[-7]"
        try:
            v = ast.literal_eval(s)
            raw = list(v) if isinstance(v, (list, tuple, set)) else [v]
        except Exception:
            # fallback: extract ints from "2,3,6" etc.
            raw = re.findall(r"-?\d+", s)

    out = []
    for r in raw:
        try:
            out.append(int(r))
        except Exception:
            pass
    return out

def format_ukb_qualifications_codes(x, treat_prefer_not_as_na=True):
    """
    Map UKB qualifications coded as a list of ints (e.g. [2,3,6]) to a 5-level category.

    Input examples:
      [1], [-7], [5,6], [2,3], [2,3,6], [-3], etc.

    Rules:
      - If any of {1,2,3,-7} appear, return highest attainment by precedence:
          1 > 2 > 3 > -7
        (i.e., College/Uni > A-levels > O-levels > None of the above)
      - Codes 4,5,6 map to "Other"
      - Code -3 ("Prefer not to answer") is treated as NA by default (configurable)
      - If list is empty/NA -> NA
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return pd.NA

    # Expect list-like; if a scalar sneaks in, coerce
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        codes = list(x)
    else:
        try:
            codes = [int(x)]
        except Exception:
            return pd.NA

    # normalise to ints where possible
    norm = []
    for c in codes:
        if c is None or (isinstance(c, float) and np.isnan(c)):
            continue
        try:
            norm.append(int(c))
        except Exception:
            continue

    if len(norm) == 0:
        return pd.NA

    # Handle prefer-not-to-answer
    if treat_prefer_not_as_na and (-3 in norm) and len(norm) == 1:
        return pd.NA

    # Map each code to a level
    levels = []
    for c in norm:
        if c == -3 and treat_prefer_not_as_na:
            continue
        levels.append(CODE_TO_LEVEL.get(c, "Other"))

    if len(levels) == 0:
        return pd.NA

    # If multiple selections, choose highest attainment (lowest precedence value)
    best = min(levels, key=lambda L: PRECEDENCE[L])
    return best

def add_qualifications_factor(df, col="edu_quals", out_col="edu_5cat",
                              treat_prefer_not_as_na=True):
    ser = df[col].map(lambda x: format_ukb_qualifications_codes(x, treat_prefer_not_as_na))
    df[out_col] = pd.Categorical(ser, categories=LEVELS, ordered=True)
    return df

In [ ]:
data["edu_quals"] = data["edu_quals"].map(parse_qual_list)
add_qualifications_factor(data, col="edu_quals", out_col="edu_quals_cat")

In [ ]:
data["proxy_birthdate"] = pd.to_datetime(dict(year=data["year_of_birth"], month=data["month_of_birth"], day=1))

In [ ]:
data.to_csv('ukb_covariates_formatted.csv.gz', compression='gzip', index=False)

In [ ]:
cmd = f"dx upload ukb_covariates_formatted.csv.gz --path {project_id}:/path/to/scratch/"

returned_value = subprocess.call(cmd, shell=True)  # returns the exit code in unix
print('Returned value:', returned_value)